In [1]:
# --------------------------------------------------
# 1. Imports (add these to the existing ones)
# --------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools
from torchvision.transforms import Resize
from ast import literal_eval
from sklearn.metrics import matthews_corrcoef
from torchvision import models
from transformers import AutoTokenizer, AutoModelForMaskedLM
import math
import json
from collections import defaultdict
import seaborn as sns
from matplotlib.colors import LogNorm
import random
# --------------------------------------------------

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [2]:
with open('ToxPre/The first level benchmark/Non-TXP.txt') as f:
    non_tox = f.read().splitlines()
    
with open('ToxPre/The first level benchmark/TXP.txt') as f:
    tox = f.read().splitlines()

non_tox = [p.strip() for p in non_tox if '>' not in p]
tox = [p.strip() for p in tox if '>' not in p]

In [3]:

df = pd.DataFrame([(s, True, False) for s in tox] + [(s, False, True) for s in non_tox], columns=['sequence', 'toxic', 'non_toxic'])

In [4]:
import random as pyrandom
# --- Data augmentation for single-function peptides (MCMFPP paper, Zhao et al. 2025) ---
def augment_single_function_peptides(df, label_columns, n_mask=2):
    """Augment single-function peptides by masking random amino acids with 'X'.
    For each single-function peptide, create n_mask masked copies.
    (Skipping flip/reverse since ESM2 expects natural-direction sequences.)
    """
    single_mask = df[label_columns].sum(axis=1) == 1
    single_df = df[single_mask]
    
    augmented_rows = []
    for _, row in single_df.iterrows():
        seq = row['sequence']
        labels = row[label_columns].values
        
        for _ in range(n_mask):
            if len(seq) > 1:
                pos = pyrandom.randint(0, len(seq) - 1)
                masked_seq = seq[:pos] + 'X' + seq[pos+1:]
                new_row = {'sequence': masked_seq}
                new_row.update({col: int(labels[i]) for i, col in enumerate(label_columns)})
                augmented_rows.append(new_row)
    
    if augmented_rows:
        aug_df = pd.DataFrame(augmented_rows)
        combined = pd.concat([df, aug_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
        print(f"  Augmented: {len(df)} → {len(combined)} ({len(single_df)} single-func × {n_mask} masks)")
        return combined
    return df




In [5]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
# Split into 80% train and 20% test
train_df, test_df = train_test_split(df, test_size=0.1, random_state=RANDOM_SEED)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# creating dataset

In [6]:
import torch
import numpy as np


import torch


# Example usage:
# seq = "ALDFR" -> Dipeptides: AL, LD, DF, FR
# vector = get_dipeptide_composition(seq)


esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")



PCP_columns = ['PCP_PC','PCP_NC','PCP_NE','PCP_PO','PCP_NP','PCP_AL','PCP_CY','PCP_AR','PCP_AC','PCP_BS','PCP_NE_pH','PCP_HB','PCP_HL','PCP_NT','PCP_HX','PCP_SC','PCP_SS_HE','PCP_SS_ST','PCP_SS_CO','PCP_SA_BU','PCP_SA_EX','PCP_SA_IN','PCP_TN','PCP_SM','PCP_LR','PCP_Z1','PCP_Z2','PCP_Z3','PCP_Z4','PCP_Z5']
PCP_DATA = {'A': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.24, -2.32, 0.6, -0.14, 1.3]),
'C': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.84, -1.67, 3.71, 0.18, -2.65]),
'D': np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 3.98, 0.93, 1.93, -2.46, 0.75]),
'E': np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 3.11, 0.26, -0.11, -3.04, -0.25]),
'F': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.22, 1.94, 1.06, 0.54, -0.62]),
'G': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 2.05, -4.06, 0.36, -0.82, -0.38]),
'H': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 2.47, 1.95, 0.26, 3.9, 0.09]),
'I': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -3.89, -1.73, -1.71, -0.84, 0.26]),
'K': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 2.29, 0.89, -2.49, 1.49, 0.31]),
'L': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.28, -1.3, -1.49, -0.72, 0.84]),
'M': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, -2.85, -0.22, 0.47, 1.94, -0.98]),
'N': np.array([0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 3.05, 1.62, 1.04, -1.15, -1.61]),
'P': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, -1.66, 0.27, 1.84, 0.7, 2.0]),
'Q': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.75, 0.5, -1.44, -1.34, 0.66]),
'R': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 3.52, 2.5, -3.5, 1.99, -0.17]),
'S': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 2.39, -1.07, 1.15, -1.39, 0.67]),
'T': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.75, -2.18, -1.12, -1.46, -0.4]),
'V': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, -2.59, -2.64, -1.54, -0.85, -0.02]),
'W': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.36, 3.94, 0.59, 3.44, -1.59]),
'Y': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, -2.54, 2.44, 0.43, 0.04, -1.47]),
'X': np.array([0.15, 0.1, 0.75, 0.25, 0.45, 0.3, 0.05, 0.15, 0.1, 0.15, 0.75, 0.5, 0.25, 0.3, 0.1, 0.1, 0.4, 0.35, 0.25, 0.4, 0.3, 0.3, 0.2, 0.45, 0.55, 0.0025000000000000356, 0.002499999999999991, 0.0019999999999999545, 0.0004999999999999877, -0.16299999999999998]),
'<pad>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
'<start>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
'<end>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
}

PCP_DATA_ID = dict()
for k, v in PCP_DATA.items():
    k = esm_tokenizer.token_to_id(k)
    PCP_DATA_ID[k] = v



def get_pcp_features(sequence):
    if isinstance(sequence, str):
        sequence = list(sequence)
        result = [PCP_DATA.get(c, PCP_DATA['X']) for c in sequence]
        result = torch.tensor(result, dtype=torch.float32)

    if isinstance(sequence, torch.Tensor):
        
        result = [PCP_DATA_ID.get(c.item(), PCP_DATA_ID[3]) for c in sequence] 
        result = torch.tensor(result, dtype=torch.float32)
    return result


In [7]:
class PeptideDataset(Dataset):
    """Pre-tokenizes all sequences at init time for much faster training."""

    def __init__(self, dataframe, tokenizer, label_columns, max_length=128):
        self.labels = dataframe[label_columns].values
        self.max_length = max_length
        
        # Pre-tokenize ALL sequences once
        sequences = dataframe['sequence'].tolist()
        encodings = tokenizer(
            sequences,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        self.input_ids = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']
        
        # Pre-compute PCP features for all sequences
        self.pcp_features = torch.stack([
            get_pcp_features(self.input_ids[i]) for i in range(len(sequences))
        ])
        
        print(f"  Pre-tokenized {len(sequences)} sequences")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            'input_ids': self.input_ids[index],
            'attention_mask': self.attention_masks[index],
            'pcp': self.pcp_features[index],
            'labels': torch.FloatTensor(self.labels[index])
        }


In [8]:
LABEL_COLUMNS = [
    'toxic'
]

MAX_LEN = 100
train_ds = PeptideDataset(train_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)
test_ds = PeptideDataset(test_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)

/tmp/ipykernel_1657227/3230906242.py:60: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  result = torch.tensor(result, dtype=torch.float32)


  Pre-tokenized 4221 sequences
  Pre-tokenized 469 sequences


In [9]:
test_ds[0]['pcp'].sum()

tensor(222.6800)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM

# =====================================================================
# Model v19 (Lite): Sequence-Level Cross-Attention + Task Query Decoder
# Parameter Count: ~19.6 Million
# =====================================================================

class ESM2_Encoder(nn.Module):
    def __init__(self, model_name, trainable=True, unfreeze_last_n=0):
        super().__init__()
        self.esm_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.esm_mlm.config.hidden_size
        if not trainable:
            for param in self.esm_mlm.parameters():
                param.requires_grad = False
            if unfreeze_last_n > 0:
                for layer in self.esm_mlm.esm.encoder.layer[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        return self.esm_mlm.esm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        w = x.mean(dim=-1)
        return x * self.fc(w).unsqueeze(-1)

class EnhancedCNN1D(nn.Module):
    def __init__(self, vocab_size=33, embed_dim=128, conv_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.branches = nn.ModuleList()
        for k in [3, 5, 7]:
            self.branches.append(nn.Sequential(
                nn.Conv1d(embed_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
                nn.Conv1d(conv_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
            ))
        self.res_proj = nn.Conv1d(embed_dim, conv_dim, 1)
        self.se = SEBlock(conv_dim * 3)
        self.dropout = nn.Dropout(0.2)
        self.hidden_size = conv_dim * 3 * 2  # 1536

    def forward(self, x):
        x = self.embed(x).transpose(1, 2)
        res = self.res_proj(x)
        outs = [branch(x) + res for branch in self.branches]
        combined = torch.cat(outs, dim=1)
        combined = self.se(combined)
        return self.dropout(torch.cat([combined.max(dim=-1).values, combined.mean(dim=-1)], dim=1))


class EnhancedPCP(nn.Module):
    """PCP encoder returning sequence-level features BxLx192."""
    def __init__(self, pcp_dim=30, conv_dim=64, hidden_size=96, num_layers=2):
        super().__init__()
        self.conv_branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(pcp_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
            ) for k in [3, 5, 7]
        ])
        self.conv_merge = nn.Sequential(
            nn.Conv1d(conv_dim * 3, conv_dim * 2, 1),
            nn.BatchNorm1d(conv_dim * 2), nn.GELU(), nn.Dropout1d(0.1)
        )
        self.gru = nn.GRU(conv_dim * 2, hidden_size, num_layers=num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=0.1 if num_layers > 1 else 0)
        self.output_dim = hidden_size * 2  # 192

    def forward(self, pcp_input, attention_mask):
        x = pcp_input.transpose(1, 2)
        x = torch.cat([b(x) for b in self.conv_branches], dim=1)
        x = self.conv_merge(x).transpose(1, 2)
        gru_out, _ = self.gru(x)
        return gru_out  # BxLx192



class MultiHeadAttentionPool(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.attn = nn.Sequential(
            nn.Linear(dim, 128), 
            nn.Tanh(), 
            nn.Linear(128, num_heads)
        )
        self._last_weights = None

    def forward(self, x, mask):
        scores = self.attn(x)
        scores = scores.masked_fill(mask.unsqueeze(-1) == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        self._last_weights = weights
        pooled = (x.unsqueeze(2) * weights.unsqueeze(-1)).sum(dim=1)
        return pooled.view(x.size(0), -1)

    # def orthogonality_loss(self):
    #     if self._last_weights is None:
    #         return 0.0
    #     w = self._last_weights.transpose(1, 2)
    #     gram = torch.bmm(w, w.transpose(1, 2))
    #     eye = torch.eye(self.num_heads, device=gram.device).unsqueeze(0)
    #     return ((gram - eye) ** 2).mean()

    def orthogonality_loss(self):
        """Penalize overlap between head attention distributions."""
        if self._last_weights is None:
            return 0.0
            
        # weights: [B, L, num_heads] -> transpose to [B, num_heads, L]
        w = self._last_weights.transpose(1, 2)  
        
        # Gram matrix of head attention distributions: shape [B, num_heads, num_heads]
        gram = torch.bmm(w, w.transpose(1, 2))  
        
        # Create a boolean mask for the off-diagonal elements (~torch.eye inverts the identity matrix)
        mask = ~torch.eye(self.num_heads, dtype=torch.bool, device=gram.device)
        
        # Penalize only the off-diagonal overlap to encourage heads to focus on different tokens.
        # We want these dot products to be pushed towards 0.
        loss = (gram[:, mask] ** 2).mean()
        
        return loss
    
class GatedFusion(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x)

class PeptideNetwork(nn.Module):
    def __init__(self, num_classes=21, mask_token_id=32):
        super().__init__()
        self.mask_token_id = mask_token_id
        self.num_classes = num_classes

        # BOTH encoders are now the 8M parameter t6 model
        self.esm_t6_a = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=True)        
        self.esm_t6_b = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=False, unfreeze_last_n=2)                                  
        self.cnn = EnhancedCNN1D()          # Bx768
        #self.pcp_model = EnhancedPCP()      # BxLx192

        # REDUCED cross_dim to 128 to save parameters
        cross_dim = 128
        self.proj_t6_a = nn.Linear(320, cross_dim)
        self.proj_t6_b = nn.Linear(320, cross_dim) # Updated to 320 for t6
        #self.proj_pcp = nn.Linear(192, cross_dim)

        self.cross_t6_a = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_a = nn.LayerNorm(cross_dim)
        
        self.cross_t6_b = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_b = nn.LayerNorm(cross_dim)
        
        # self.cross_pcp = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        # self.ln_ca_pcp = nn.LayerNorm(cross_dim)

        self.pool_t6_a = MultiHeadAttentionPool(cross_dim, num_heads=4)
        self.pool_t6_b = MultiHeadAttentionPool(cross_dim, num_heads=4)
        # self.pool_pcp = MultiHeadAttentionPool(cross_dim, num_heads=4)

        # Bottleneck reduction layer before fusion
        # Concat size: 128*4*3 (pools) + 768 (CNN) = 1536 + 768 = 2304
        concat_size = cross_dim * 4 * 2 + self.cnn.hidden_size
        
        reduced_dim_size = 512
        self.dim_reduce = nn.Sequential(
            nn.Linear(concat_size, reduced_dim_size),
            nn.GELU()
        )
        
        # Fusion now operates efficiently on 512 dimensions
        self.fusion = GatedFusion(reduced_dim_size)
        self.ln = nn.LayerNorm(reduced_dim_size)

        # Task Query Decoder
        self.task_dim = 128
        self.n_memory_tokens = 16
        self.memory_proj = nn.Sequential(
            nn.Linear(reduced_dim_size, self.task_dim * self.n_memory_tokens),
            nn.GELU(),
        )
        self.task_queries = nn.Embedding(num_classes, self.task_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.task_dim, nhead=4, dim_feedforward=256,
            batch_first=True, dropout=0.1
        )
        self.task_decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.task_classifiers = nn.ModuleList([
            nn.Linear(self.task_dim, 1) for _ in range(num_classes)
        ])

    def _mask_tokens(self, input_ids, attention_mask, mask_prob=0.15):
        masked_ids = input_ids.clone()
        prob_matrix = torch.full_like(input_ids, mask_prob, dtype=torch.float)
        prob_matrix[attention_mask == 0] = 0
        prob_matrix[:, 0] = 0
        seq_lens = attention_mask.sum(dim=1)
        for i in range(len(seq_lens)):
            if seq_lens[i] > 1:
                prob_matrix[i, seq_lens[i] - 1] = 0
        mask = torch.bernoulli(prob_matrix).bool()
        masked_ids[mask] = self.mask_token_id
        return masked_ids

    def _extract_features(self, seq_input, seq_mask, pcp_input):
        esm6_a_seq = self.esm_t6_a(seq_input, seq_mask)       
        esm6_b_seq = self.esm_t6_b(seq_input, seq_mask)     
        cnn_feat = self.cnn(seq_input)                          

        t6_a = self.proj_t6_a(esm6_a_seq)       
        t6_b = self.proj_t6_b(esm6_b_seq)    


        kv_pad = seq_mask == 0  

        ca_t6_a, _ = self.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
        ca_t6_a = self.ln_ca_t6_a(t6_a + ca_t6_a)

        ca_t6_b, _ = self.cross_t6_b(t6_b, t6_a, t6_a, key_padding_mask=kv_pad)
        ca_t6_b = self.ln_ca_t6_b(t6_b + ca_t6_b)


        pooled_t6_a = self.pool_t6_a(ca_t6_a, seq_mask)
        pooled_t6_b = self.pool_t6_b(ca_t6_b, seq_mask)

        # Concat -> Reduce -> Fuse
        combined = torch.cat([pooled_t6_a, pooled_t6_b, cnn_feat], dim=1)  
        reduced = self.dim_reduce(combined)
        final_fusion = self.ln(reduced + self.fusion(reduced))
        
        return final_fusion

    def _classify(self, final_fusion):
        B = final_fusion.size(0)
        memory = self.memory_proj(final_fusion).view(B, self.n_memory_tokens, self.task_dim)
        tgt = self.task_queries.weight.unsqueeze(0).expand(B, -1, -1)
        decoded = self.task_decoder(tgt, memory)
        logits = torch.cat([self.task_classifiers[i](decoded[:, i, :])
                           for i in range(self.num_classes)], dim=1)
        return logits

    def forward(self, seq_input, seq_mask, pcp_input, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        final_fusion = self._extract_features(seq_input, seq_mask, pcp_input)
        return self._classify(final_fusion)

    def ortho_loss(self):
        return (self.pool_t6_a.orthogonality_loss() +
                self.pool_t6_b.orthogonality_loss() 
                ) / 2
    
    def get_features(self, seq_input, seq_mask, pcp_input, mask_tokens=False, mask_prob=0.15):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask, mask_prob)
        return self._extract_features(seq_input, seq_mask, pcp_input)

    def classify_features(self, combined):
        return self._classify(combined)

In [11]:
import copy

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        # Update parameters (weights & biases)
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)
        
        # FIX: Update only floating point buffers (skip int buffers like num_batches_tracked)
        for s_buf, m_buf in zip(self.shadow.buffers(), model.buffers()):
            if s_buf.dtype.is_floating_point:
                s_buf.data.mul_(self.decay).add_(m_buf.data, alpha=1 - self.decay)
            else:
                # For integer buffers, just copy directly (no EMA smoothing needed)
                s_buf.data.copy_(m_buf.data)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)


In [12]:
DEVICE    = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [13]:

FUNC_INDICES = [
    0]
index_endpoint = {0: 'toxic'}

In [14]:
epochs = 30
BATCH_SIZE = 32

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [16]:
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, jaccard_score
import json

In [21]:
# --- HELPER FUNCTIONS ---
from sklearn.model_selection import KFold
import copy
import json
import torch
import torch.nn.functional as F

def find_optimal_threshold(y_true, y_pred_probs):
    y_true_cls = y_true[:, 0]
    best_acc, best_t = -1, 0.5
    p_pos = y_pred_probs[:, 0]
    for t in np.arange(0.01, 1.0, 0.03):
        pred_cls = (p_pos >= t).astype(int)
        acc = accuracy_score(y_true_cls, pred_cls)
        if acc > best_acc: 
            best_acc = acc
            best_t = t
    return best_t

def calculate_complete_metrics(y_true, y_pred_probs, threshold, class_names):
    y_true_cls = y_true[:, 0]
    p_pos = y_pred_probs[:, 0]
    y_pred_cls = (p_pos >= threshold).astype(int)

    mcc = matthews_corrcoef(y_true_cls, y_pred_cls)
    acc = accuracy_score(y_true_cls, y_pred_cls)
    prec = precision_score(y_true_cls, y_pred_cls, zero_division=0)
    rec = recall_score(y_true_cls, y_pred_cls, zero_division=0)
    f1 = f1_score(y_true_cls, y_pred_cls, zero_division=0)
    jaccard = jaccard_score(y_true_cls, y_pred_cls)
    
    # Avoid warnings if a batch/fold only has 1 class
    if len(np.unique(y_true_cls)) > 1:
        auc_val = roc_auc_score(y_true_cls, p_pos)
    else:
        auc_val = 0.5
        
    tn, fp, fn, tp = confusion_matrix(y_true_cls, y_pred_cls, labels=[0,1]).ravel()
    results = {
        class_names[0]: {
            "mcc": float(mcc), "threshold": float(threshold),
            "accuracy": float(acc), "precision": float(prec),
            "recall": float(rec), "f1_score": float(f1), "roc_auc": float(auc_val),
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
        }
    }
    return results, float(acc), float(mcc), float(jaccard)

def calculate_mcc(tp, tn, fp, fn):
    num = (tp * tn) - (fp * fn)
    den_sq = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / math.sqrt(den_sq) if den_sq > 0 else 0.0

def feature_mixup(features, labels, alpha=0.4):
    batch_size = features.size(0)
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(batch_size, device=features.device)
    mixed_features = lam * features + (1 - lam) * features[index]
    mixed_labels = lam * labels + (1 - lam) * labels[index]
    return mixed_features, mixed_labels

def rdrop_kl_loss(logits1, logits2):
    kl1 = F.binary_cross_entropy_with_logits(logits1, torch.sigmoid(logits2).detach(), reduction='mean')
    kl2 = F.binary_cross_entropy_with_logits(logits2, torch.sigmoid(logits1).detach(), reduction='mean')
    return (kl1 + kl2) / 2


def update_swa(model_state, swa_state, n):
    if swa_state is None:
        return {k: v.clone() for k, v in model_state.items()}, 1
    for k in swa_state:
        swa_state[k] = (swa_state[k] * n + model_state[k]) / (n + 1)
    return swa_state, n + 1

import torch.nn as nn

def enable_dropout(model):
    """
    Selectively enables dropout-related layers for Test-Time Augmentation (TTA)
    while keeping BatchNorm layers strictly frozen in eval() mode.
    """
    for module in model.modules():
        # 1. Catch all standard PyTorch dropouts (Dropout, Dropout1d, Dropout2d, etc.)
        # This will also naturally catch the Dropouts inside the Hugging Face ESM models
        if isinstance(module, nn.modules.dropout._DropoutNd):
            module.train()
            
        # 2. Catch complex modules that handle their own internal dropout
        elif isinstance(module, (
            nn.MultiheadAttention, 
            nn.TransformerDecoderLayer, 
            nn.TransformerDecoder, 
            nn.GRU
        )):
            module.train()
            
        # 3. Fallback catch just in case Hugging Face updates their backend 
        # to use a custom dropout class name that doesn't inherit from _DropoutNd
        elif "Dropout" in module.__class__.__name__:
            module.train()

# --- 10-FOLD CROSS VALIDATION SETUP ---
from torch.utils.data import Subset, DataLoader

kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
fold_results = []
history_log_path = "toxpre_10fold.json"

tta_passes = 5
mixup_alpha = 0.3
rdrop_alpha = 1
swa_start_epoch = 18

print("=" * 70)
print(f"Starting 10-Fold Cross Validation | Epochs: {epochs}")
print("=" * 70)

for fold, (train_idx, val_idx) in enumerate(kf.split(train_ds)):
    print(f"\n{'*'*20} FOLD {fold+1}/10 {'*'*20}")
    
    # Shuffle train_val_idx before splitting to avoid class imbalance 

    fold_train_ds = Subset(train_ds, train_idx)
    fold_val_ds = Subset(train_ds, val_idx)

    train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(fold_val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # Reinitialize Model
    torch.cuda.empty_cache()
    model = PeptideNetwork(num_classes=1, mask_token_id=32).to(DEVICE)
    ema = EMA(model, decay=0.999)
    criterion = nn.BCEWithLogitsLoss()

    # Optimizer — 3 LR groups (Lowered LRs to prevent early overfitting)
    esm_t6_ids = set(id(p) for p in model.esm_t6_a.esm_mlm.parameters())
    esm_t6_b_ids = set(id(p) for p in model.esm_t6_b.esm_mlm.parameters())

    esm_t6_params = [p for p in model.parameters() if id(p) in esm_t6_ids and p.requires_grad]
    esm_t6_b_params = [p for p in model.parameters() if id(p) in esm_t6_b_ids and p.requires_grad]
    other_params = [p for p in model.parameters() if id(p) not in esm_t6_ids and id(p) not in esm_t6_b_ids and p.requires_grad]

    optimizer = optim.AdamW([
        {'params': esm_t6_params, 'lr': 1e-5},     # Reduced from 1e-5
        {'params': esm_t6_b_params, 'lr': 2e-6},   # Reduced from 2e-6
        {'params': other_params, 'lr': 1e-4},      # Reduced from 1e-4
    ], weight_decay=0.05)     

    steps_per_epoch = len(train_loader)
    warmup_epochs = 3
    total_steps = epochs * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_epochs * steps_per_epoch:
            return step / max(1, warmup_epochs * steps_per_epoch)
        progress = (step - warmup_epochs * steps_per_epoch) / max(1, total_steps - warmup_epochs * steps_per_epoch)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler('cuda')

    best_val_accuracy = 0.0
    patience_counter = 0
    early_stop_patience = 7
    best_model_path_fold = f"toxpre_fold_{fold+1}.pt"
    
    swa_model = None
    swa_n = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0

        for data in train_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            pcp = data['pcp'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                features1 = model.get_features(input_ids, attention_mask, pcp, mask_tokens=True, mask_prob=0.15)
                features2 = model.get_features(input_ids, attention_mask, pcp, mask_tokens=True, mask_prob=0.15)

                logits1_clean = model.classify_features(features1)
                logits2_clean = model.classify_features(features2)

                # R-Drop consistency
                rdrop_loss = rdrop_kl_loss(logits1_clean, logits2_clean)

                # Mixup on first pass
                mixed_features, mixed_labels = feature_mixup(features1, labels, alpha=mixup_alpha)
                logit_mixed =  model.classify_features(mixed_features)

                # ASL loss
                asl_loss1 = criterion(logit_mixed, mixed_labels)
                asl_loss2 = criterion(logits2_clean, labels)

            

                # Orthogonality loss on multi-head attention pools
                ortho = model.ortho_loss()

                loss = (asl_loss1 + asl_loss2) / 2 + rdrop_alpha * rdrop_loss + 0.1 * ortho


            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ema.update(model)

            total_train_loss += loss.item() * input_ids.size(0)

        # SWA Logic
        if epoch >= swa_start_epoch:
            swa_model, swa_n = update_swa(ema.shadow.state_dict(), swa_model, swa_n)

        if swa_model is not None and epoch >= swa_start_epoch:
            eval_model = copy.deepcopy(ema.shadow)
            eval_model.load_state_dict(swa_model)
            eval_model.eval()
        else:
            eval_model = ema.shadow
            eval_model.eval()

        # Validation
        spec_y_val = []
        tta_preds = [[] for _ in range(tta_passes)]
        with torch.no_grad():
            for data in val_loader:
                input_ids = data['input_ids'].to(DEVICE)
                attention_mask = data['attention_mask'].to(DEVICE)
                pcp = data['pcp'].to(DEVICE)
                labels = data['labels'].to(DEVICE)
                
                with torch.amp.autocast('cuda'):
                    logits = eval_model(input_ids, attention_mask, pcp)

                
                spec_y_val.append(labels.cpu().numpy())
                tta_preds[0].append(torch.sigmoid(logits).detach().cpu().numpy())

                #eval_model.train()
                enable_dropout(eval_model)
                for t in range(1, tta_passes):
                    with torch.amp.autocast('cuda'):
                        logits_t = eval_model(input_ids, attention_mask, pcp)
                    tta_preds[t].append(torch.sigmoid(logits_t).detach().cpu().numpy())
                eval_model.eval()

        y_true_val = np.concatenate(spec_y_val)
        y_pred_spec_val = np.mean([
            np.concatenate(tta_preds[t]) for t in range(tta_passes)
        ], axis=0)


        val_thresh = find_optimal_threshold(y_true_val, y_pred_spec_val)
        _, val_acc, _, val_jaccard = calculate_complete_metrics(y_true_val, y_pred_spec_val, val_thresh, ['toxic'])

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            best_val_thresh = val_thresh
            patience_counter = 0
            torch.save(eval_model.state_dict(), best_model_path_fold)
        else:
            patience_counter += 1

        print(f"Fold {fold+1} | Ep {epoch:02d}/{epochs} | Val Acc: {val_acc:.4f} | Val Jac: {val_jaccard} | Best Val jac: {best_val_accuracy:.4f}")

        if patience_counter >= early_stop_patience:
            print(f"Early stopping at epoch {epoch}!")
            break

        if swa_model is not None and epoch >= swa_start_epoch:
            del eval_model
    
    # ----------------------------------------------------
    # FINAL TEST EVALUATION FOR THIS FOLD
    # ----------------------------------------------------
    print(f"Evaluating Fold {fold+1} Test Set...")
    best_fold_model = PeptideNetwork(num_classes=1, mask_token_id=32).to(DEVICE)
    best_fold_model.load_state_dict(torch.load(best_model_path_fold))
    best_fold_model.eval()

    spec_y_test = []
    tta_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for test_data in test_loader:
            input_ids = test_data['input_ids'].to(DEVICE)
            attention_mask = test_data['attention_mask'].to(DEVICE)
            pcp = test_data['pcp'].to(DEVICE)
            labels = test_data['labels'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = best_fold_model(input_ids, attention_mask, pcp)
                loss_val = criterion(logits, labels)
            spec_y_test.append(labels.cpu().numpy())
            tta_preds[0].append(torch.sigmoid(logits).detach().cpu().numpy())

            #best_fold_model.train()
            enable_dropout(best_fold_model)
            for t in range(1, tta_passes):
                with torch.amp.autocast('cuda'):
                    logits_t = best_fold_model(input_ids, attention_mask, pcp)
                tta_preds[t].append(torch.sigmoid(logits_t).detach().cpu().numpy())
            best_fold_model.eval()

    y_true_spec_test = np.concatenate(spec_y_test)
    y_pred_spec_test = np.mean([
        np.concatenate(tta_preds[t]) for t in range(tta_passes)
    ], axis=0)

    test_metrics_f, test_acc_f, test_mcc_f, _ = calculate_complete_metrics(y_true_spec_test, y_pred_spec_test, best_val_thresh, ['toxic'])
    
    print(f"Fold {fold+1} Results -> Acc: {test_acc_f:.4f} | MCC: {test_mcc_f:.4f} | AUC: {test_metrics_f['toxic']['roc_auc']:.4f}")
    
    fold_results.append({
        "fold": fold + 1,
        "best_val_thresh": float(best_val_thresh),
        "test_acc": test_acc_f,
        "test_mcc": test_mcc_f,
        "metrics": test_metrics_f
    })

    with open(history_log_path, "w") as f:
        json.dump(fold_results, f, indent=2)

# Global Cross Validation Summary
print("\n" + "="*70)
print("10-FOLD CROSS Validation SUMMARY:")
accs = [r['test_acc'] for r in fold_results]
mccs = [r['test_mcc'] for r in fold_results]

print(f"Average Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Average MCC:      {np.mean(mccs):.4f} ± {np.std(mccs):.4f}")
print("="*70)

Starting 10-Fold Cross Validation | Epochs: 30

******************** FOLD 1/10 ********************
Fold 1 | Ep 01/30 | Val Acc: 0.6099 | Val Jac: 0.4463087248322148 | Best Val jac: 0.6099
Fold 1 | Ep 02/30 | Val Acc: 0.7116 | Val Jac: 0.5271317829457365 | Best Val jac: 0.7116
Fold 1 | Ep 03/30 | Val Acc: 0.7612 | Val Jac: 0.5910931174089069 | Best Val jac: 0.7612
Fold 1 | Ep 04/30 | Val Acc: 0.7991 | Val Jac: 0.6755725190839694 | Best Val jac: 0.7991
Fold 1 | Ep 05/30 | Val Acc: 0.8274 | Val Jac: 0.7159533073929961 | Best Val jac: 0.8274
Fold 1 | Ep 06/30 | Val Acc: 0.8440 | Val Jac: 0.7480916030534351 | Best Val jac: 0.8440
Fold 1 | Ep 07/30 | Val Acc: 0.8440 | Val Jac: 0.7441860465116279 | Best Val jac: 0.8440
Fold 1 | Ep 08/30 | Val Acc: 0.8511 | Val Jac: 0.7586206896551724 | Best Val jac: 0.8511
Fold 1 | Ep 09/30 | Val Acc: 0.8558 | Val Jac: 0.7626459143968871 | Best Val jac: 0.8558
Fold 1 | Ep 10/30 | Val Acc: 0.8605 | Val Jac: 0.76953125 | Best Val jac: 0.8605
Fold 1 | Ep 11/30 

In [ ]:
weight decay - Acc: 0.8721

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 1e-05
    lr: 4.453781512605043e-06
    maximize: False
    weight_decay: 0.05

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 2e-06
    lr: 8.907563025210084e-07
    maximize: False
    weight_decay: 0.05

Parameter Group 2
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 4.453781512605042e-05
    maximize: False
    weight_decay: 0.05
)

In [ ]:
y_true_cls = y_true_spec_test[:, 0]
p_pos = y_pred_spec_test[:, 0]
y_pred_cls = (p_pos >= best_val_thresh).astype(int)

mcc = matthews_corrcoef(y_true_cls, y_pred_cls)
acc = accuracy_score(y_true_cls, y_pred_cls)

In [ ]:
acc

0.8507462686567164

In [ ]:
best_val_thresh

np.float64(0.87)